In [1]:
import os
import sys
import argparse

import numpy as np
import matplotlib.pyplot as plt

import torch

import dolfinx
import dolfinx.fem.petsc
import ufl
from mpi4py import MPI
import basix.ufl

repo_path = os.path.abspath(os.path.join(os.getcwd(), "../../"))
sys.path.append(repo_path)

from utils import load_yaml, load_pkl, load_npy, format_elapsed_time, timing, plot_complex_valued_function, plot_real_valued_function, evaluate_expression

from scifem import create_real_functionspace

import torch
import torch.nn as nn
import torch.nn.init as init
import numpy as np
from petsc4py import PETSc
import scifem


from utils import project, norm_L2, convert_petsc_mat_to_torch_sparse_coo_tensor, convert_weight_to_tensor
from typing import Optional
import pickle
import time
from tqdm import tqdm
import matplotlib.ticker as ticker

from data_generation.differential_equations import ElasticityLeastSquares

----------------------------------------
2025-11-28 00:05:51 - Start Program
----------------------------------------


In [2]:
mesh_args = {
    "lower_left_x": 0.0,        # x-coordinate of lower-left corner
    "lower_left_y": 0.0,        # y-coordinate of lower-left corner
    "upper_right_x": 2.0,       # x-coordinate of upper-right corner
    "upper_right_y": 1.0,       # y-coordinate of upper-right corner
    "num_x": 256,               # number of cells along x-axis
    "num_y": 128,                # number of cells along y-axis
    "mesh_cell_type": "triangle"   # "triangle" or "quadrilateral"
}


In [3]:
function_space_args = {
    "p": {
        "family": "CG",
        "degree": 1
    },
    "u": {
        "family": "CG",
        "degree": 2
    },
    "sigma": {
        "family": "RT",
        "degree": 2
    },
    "w": {
        "family": "CG",
        "degree": 2
    },
    "q": {
        "family": "CG",
        "degree": 2
    }
}


In [4]:
function_space_finer_args = {
    "p": {
        "family": "CG",
        "degree": 1
    },
    "u": {
        "family": "CG",
        "degree": 4
    },
    "sigma": {
        "family": "RT",
        "degree": 4
    },
    "w": {
        "family": "CG",
        "degree": 4
    },
    "q": {
        "family": "CG",
        "degree": 4
    }
}


In [5]:
print(f'mesh args: {mesh_args}')
print(f'function space args: {function_space_args}')
print(f'function space finer args: {function_space_finer_args}')

mesh args: {'lower_left_x': 0.0, 'lower_left_y': 0.0, 'upper_right_x': 2.0, 'upper_right_y': 1.0, 'num_x': 256, 'num_y': 128, 'mesh_cell_type': 'triangle'}
function space args: {'p': {'family': 'CG', 'degree': 1}, 'u': {'family': 'CG', 'degree': 2}, 'sigma': {'family': 'RT', 'degree': 2}, 'w': {'family': 'CG', 'degree': 2}, 'q': {'family': 'CG', 'degree': 2}}
function space finer args: {'p': {'family': 'CG', 'degree': 1}, 'u': {'family': 'CG', 'degree': 4}, 'sigma': {'family': 'RT', 'degree': 4}, 'w': {'family': 'CG', 'degree': 4}, 'q': {'family': 'CG', 'degree': 4}}


In [6]:
elasticity_least_squares = ElasticityLeastSquares(mesh_args, function_space_args)
elasticity_least_squares_finer = ElasticityLeastSquares(mesh_args, function_space_finer_args)

In [7]:
mesh = elasticity_least_squares.mesh
Vh = elasticity_least_squares.Vh

In [8]:
# Make sure two function spaces are defined on the same mesh
sigma_element_component = basix.ufl.element(family=function_space_finer_args["sigma"]["family"], 
                                            cell=mesh_args["mesh_cell_type"], 
                                            degree=function_space_finer_args["sigma"]["degree"])
u_element_component = basix.ufl.element(family=function_space_finer_args["u"]["family"], 
                                        cell=mesh_args["mesh_cell_type"], 
                                        degree=function_space_finer_args["u"]["degree"])
sigma_element = basix.ufl.mixed_element([sigma_element_component, sigma_element_component])
u_element = basix.ufl.mixed_element([u_element_component, u_element_component])
sigma_u_element = basix.ufl.mixed_element([sigma_element, u_element])

finer_Vh = {
    'sigma_u': dolfinx.fem.functionspace(mesh, sigma_u_element)
}

In [9]:
num_samples = 100

In [10]:
train_dataset_path = repo_path + "/results/elasticity/train_dataset"
test_dataset_path = repo_path + "/results/elasticity/test_dataset"

In [11]:
if mesh_args['num_x'] == 128:
    p_dof = np.load(test_dataset_path + "/p_dof.npy")[:num_samples]
elif mesh_args['num_x'] == 256:
    p_dof = np.load(test_dataset_path + "/p_dof_256x128.npy")[:num_samples]

In [12]:
elasticity_least_squares.q = elasticity_least_squares.solve_q()
elasticity_least_squares.w = elasticity_least_squares.solve_w()

In [13]:
dtype = 'float64'
sigma_u_dim = len(dolfinx.fem.Function(Vh['sigma_u']).x.array)
sigma_u_dof = np.zeros((num_samples, sigma_u_dim), dtype=dtype)

time_for_solving_PDEs = []
for i in tqdm(range(num_samples)):
    start_time = time.time()
    p = dolfinx.fem.Function(elasticity_least_squares.Vh['p'], dtype=dtype)
    p.x.array[:] = p_dof[i,:]
    sigma_u = elasticity_least_squares.solve_sigma_u(p=p)
    sigma_u_dof[i,:] = sigma_u.x.array
    end_time = time.time()
    time_for_solving_PDEs.append(end_time - start_time)

  1%|          | 1/100 [00:20<33:02, 20.03s/it]


KeyboardInterrupt: 

In [ ]:
print(f'Solved for {num_samples} samples in {np.sum(time_for_solving_PDEs):.2f} seconds')
print(f'average time per sample: {np.mean(time_for_solving_PDEs):.2f} seconds')

Solved for 100 samples in 1452.30 seconds
average time per sample: 14.52 seconds


In [ ]:
dtype = 'float64'
sigma_u_finer_dim = len(dolfinx.fem.Function(finer_Vh['sigma_u']).x.array)
sigma_u_finer_dof = np.zeros((num_samples, sigma_u_finer_dim), dtype=dtype)
sigma_u_finer_dof[:50] = np.load('/work/yuan/VCNO2/results/elasticity/test_dataset/sigma_u_dof_finer_256x128_0_50.npy')
sigma_u_finer_dof[50:] = np.load('/work/yuan/VCNO2/results/elasticity/test_dataset/sigma_u_dof_finer_256x128_50_100.npy')

In [ ]:
# elasticity_least_squares_finer.q = elasticity_least_squares_finer.solve_q()
# elasticity_least_squares_finer.w = elasticity_least_squares_finer.solve_w()

In [ ]:
# dtype = 'float64'
# sigma_u_finer_dim = len(dolfinx.fem.Function(finer_Vh['sigma_u']).x.array)
# sigma_u_finer_dof = np.zeros((num_samples, sigma_u_finer_dim), dtype=dtype)

# time_for_solving_PDEs_finer = []
# for i in tqdm(range(num_samples)):
#     start_time = time.time()
#     p_finer = dolfinx.fem.Function(elasticity_least_squares_finer.Vh['p'], dtype=dtype)
#     p_finer.x.array[:] = p_dof[i,:]
#     sigma_u_finer = elasticity_least_squares_finer.solve_sigma_u(p=p_finer)
#     sigma_u_finer_dof[i,:] = sigma_u_finer.x.array
#     end_time = time.time()
#     time_for_solving_PDEs_finer.append(end_time - start_time)

100%|██████████| 100/100 [42:49<00:00, 25.69s/it]


In [ ]:
# print(f'Solved for {num_samples} samples in {np.sum(time_for_solving_PDEs_finer):.2f} seconds')
# print(f'average time per sample: {np.mean(time_for_solving_PDEs_finer):.2f} seconds')

Solved for 100 samples in 2569.15 seconds
average time per sample: 25.69 seconds


In [ ]:
# sigma_u_dof = np.load(test_dataset_path + "/sigma_u_dof.npy")
# sigma_u_dof_finer = np.load(test_dataset_path + "/sigma_u_dof_finer.npy")
# p_dof = np.load(test_dataset_path + "/p_dof.npy")

# test_index = 1
# sigma_u_fc = dolfinx.fem.Function(Vh['sigma_u'])
# sigma_u_fc.x.array[:] = sigma_u_dof[test_index]
# sigma_fc = sigma_u_fc.sub(0).collapse()
# sigma1_fc, sigma2_fc = ufl.split(sigma_fc)
# sigma_fc_ = ufl.as_vector((sigma1_fc, sigma2_fc))
# u_fc = sigma_u_fc.sub(1).collapse()

# sigma_u_fc_finer = dolfinx.fem.Function(finer_Vh['sigma_u'])
# sigma_u_fc_finer.x.array[:] = sigma_u_dof_finer[test_index]
# sigma_fc_finer = sigma_u_fc_finer.sub(0).collapse()
# sigma1_fc_finer, sigma2_fc_finer = ufl.split(sigma_fc_finer)
# sigma_fc_finer_ = ufl.as_vector((sigma1_fc_finer, sigma2_fc_finer))
# u_fc_finer = sigma_u_fc_finer.sub(1).collapse()

In [ ]:
if mesh_args['num_x'] == 128:
    mean_p_dof = np.load(test_dataset_path + "/mean_p_dof.npy")
elif mesh_args['num_x'] == 256:
    mean_p_dof = np.load(test_dataset_path + "/mean_p_dof_256x128.npy")
mean_p_fc = dolfinx.fem.Function(Vh['p'])
mean_p_fc.x.array[:] = mean_p_dof

In [ ]:
compute_squared_hdiv_h1_norm = elasticity_least_squares.compute_squared_hdiv_h1_norm

In [ ]:
# squared_error = compute_squared_hdiv_h1_norm(sigma_fc_ - sigma_fc_finer_, u_fc - u_fc_finer, mean_p_fc)

In [ ]:
torch_dtype = {
    'float32': torch.float32,
    'float64': torch.float64,
}

In [ ]:
squared_error_list = []
reference_loss_list = []
time_for_assembling_weight = []
for test_index in tqdm(range(num_samples)):
    sigma_u_fc = dolfinx.fem.Function(Vh['sigma_u'])
    sigma_u_fc.x.array[:] = sigma_u_dof[test_index]
    sigma_fc = sigma_u_fc.sub(0).collapse()
    sigma1_fc, sigma2_fc = ufl.split(sigma_fc)
    sigma_fc_ = ufl.as_vector((sigma1_fc, sigma2_fc))
    u_fc = sigma_u_fc.sub(1).collapse()

    sigma_u_finer_fc = dolfinx.fem.Function(finer_Vh['sigma_u'])
    sigma_u_finer_fc.x.array[:] = sigma_u_finer_dof[test_index]
    sigma_finer_fc = sigma_u_finer_fc.sub(0).collapse()
    sigma1_finer_fc, sigma2_finer_fc = ufl.split(sigma_finer_fc)
    sigma_finer_fc_ = ufl.as_vector((sigma1_finer_fc, sigma2_finer_fc))
    u_finer_fc = sigma_u_finer_fc.sub(1).collapse()

    squared_error = compute_squared_hdiv_h1_norm(sigma_fc_ - sigma_finer_fc_, u_fc - u_finer_fc, mean_p_fc)

    p_fc = dolfinx.fem.Function(Vh['p'])  
    p_fc.x.array[:] = p_dof[test_index]

    start_time = time.time()
    weight = elasticity_least_squares.compute_weight(p_fc)
    end_time = time.time()
    time_for_assembling_weight.append(end_time - start_time)

    weight_tensor = convert_weight_to_tensor(weight, dtype=torch_dtype['float64'])

    y = sigma_u_dof[test_index]
    y = torch.tensor(y, dtype=torch_dtype['float64'])
    reference_loss = torch.dot(y, weight_tensor['A00'] @ y) + 2*torch.dot(y, weight_tensor['A01'])  + weight_tensor['A11']

    squared_error_list.append(squared_error)
    reference_loss_list.append(reference_loss.item())

  0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
print(f'# DoFs of sigma_u: {sigma_u_dof.shape[1]}')
print(f'# DoFs of finer sigma_u: {sigma_u_finer_dof.shape[1]}')
print(f'average time for assembling weight: {np.round(np.mean(time_for_assembling_weight), decimals=3)} seconds')
print(f'mean squared error between coarse and finer solutions: {np.mean(squared_error_list):.2e}')
print(f'mean reference loss: {np.mean(reference_loss_list):.2e}')

# DoFs of sigma_u: 263682
# DoFs of finer sigma_u: 3414018
average time for assembling weight: 0.771 seconds
mean squared error between coarse and finer solutions: 5.96e-03
mean reference loss: 2.46e-03
